## 1. Data Collection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import plotly.express as px


In [ ]:
import yfinance as yf

In [ ]:
bitcoin = yf.download(tickers='BTC-USD', start = '2017-01-01', end = '2025-12-31', interval= '1d')

In [ ]:
bitcoin = bitcoin.reset_index()

In [ ]:
bitcoin.columns = bitcoin.columns.droplevel(1)

In [ ]:
bitcoin.columns.name = None

In [ ]:
bitcoin

In [ ]:
bitcoin.shape

In [ ]:
bitcoin['Date'].max()

In [ ]:
bitcoin['Date'].min()

## 2. OHLC Chart Analysis

In [ ]:
bitcoin.dtypes

In [ ]:
bitcoin.isnull().sum()

In [ ]:
bitcoin.duplicated().sum()


In [ ]:
bitcoin = bitcoin.sort_values(by =['Date'])

In [ ]:
data = bitcoin.copy()

In [ ]:
bitcoin_Sample = data[0:100]

In [ ]:
import plotly.graph_objs as go


In [ ]:
trace = go.Candlestick( x = bitcoin_Sample['Date'],
               high = bitcoin_Sample['High'],
               open = bitcoin_Sample['Open'],
               close = bitcoin_Sample['Close'],
               low = bitcoin_Sample['Low']
    
)

In [ ]:
candle_data = [trace]

layout = {
    'title' : 'Bitcoin Historical Price',
    'xaxis': {'title' : 'Date'}
}

In [ ]:
fig = go.Figure(data= candle_data, layout = layout)

In [ ]:
fig

In [ ]:
fig.update_layout(xaxis_rangeslider_visible = False)

## 3. Is Bitcoin Risky 

In [ ]:
data['daily_return'] = data['Close'].pct_change()

In [ ]:
data.head(2)

In [ ]:
data['daily_return'].std()

In [ ]:
data['volatility_30d'] = data['daily_return'].rolling(30).std()
data['volatility_90d'] = data['daily_return'].rolling(90).std()

In [ ]:
data.columns

In [ ]:
px.line(data, 
       x = 'Date',
       y = ['volatility_30d', 'volatility_90d'],
       title = 'Bitcoin Volatility Over Time',
       labels = {
           'value' : 'Volatility(risk)',
           'variable' : 'window'
       })

In [ ]:
data[['volatility_30d', 'volatility_90d']].describe().T

## 4. How Much Would 1 Euro in Bitcoin Be Worth Today

In [ ]:
data['cumulative_return'] = (1 + data['daily_return']).cumprod()

In [ ]:
investment = 1

In [ ]:
data['investment_value'] = investment * data['cumulative_return']

In [ ]:
data.columns

In [ ]:
data[['Date', 'investment_value']]

In [ ]:
final_value = data['investment_value'].iloc[-1]

In [ ]:
px.line(data,
       x = 'Date',
       y = 'investment_value',
       title = '1 Euro Investment Growth in Bitcoin',
       labels = {
           'investment_value' : 'Investment value (Euro)'
       }
       )

## 5. Bitcoin Crash Analysis-Worst Days in History

In [ ]:
worst_days = data.nsmallest(10, 'daily_return')[['Date', 'Close', 'daily_return']]

In [ ]:
worst_days

In [ ]:
px.bar(worst_days,
       x = 'Date',
       y = 'daily_return',
       title = 'Bitcoin-Worst Crash Days in History'
)

In [ ]:
worst_days['crash_label'] = 'BTC |' + worst_days['Date'].dt.strftime('%Y-%m-%d')

In [ ]:
fig = px.bar(worst_days,
       x = 'crash_label',
       y = 'daily_return',
       title = 'Bitcoin-Worst Crash Days in History',
       labels = {
           'daily_return' : 'Daily Return(Loss)',
           'crash_label' : 'Crash Date'
       }
)
fig.add_hline(y = 0)
fig.show()

## 6. Does Volume Confirm Bitcoin Price Moves 

In [ ]:
data.columns

In [ ]:
data['price_change'] = data['Close'] - data['Open']

In [ ]:
data['Volume']

In [ ]:
data['Volume_change'] = data['Volume'].pct_change()

In [ ]:
def classify_volume_price(row):
    if row['price_change'] > 0 and row['Volume_change']> 0:
        return 'Strong Bullish'
    elif row['price_change'] < 0 and row['Volume_change']> 0:
        return 'Strong Bearish'
    elif row['price_change'] > 0 and row['Volume_change']< 0:
        return 'Weak Bullish'
    else:
        return 'Weak Bearish'
    

In [ ]:
data['volume_price_signal'] = data.apply(classify_volume_price, axis =1)

In [ ]:
data['volume_price_signal'].value_counts()

## 7. Best & Worst Years to Invest in Bitcoin 

In [ ]:
data.head(1)

In [ ]:
data['Year'] = data['Date'].dt.year

In [ ]:
yearly_returns = data.groupby('Year')['daily_return'].apply(lambda x : (1+x).prod() - 1)

In [ ]:
yearly_returns.idxmax()

In [ ]:
yearly_returns.idxmin()

In [ ]:
yearly_df = yearly_returns.reset_index()
yearly_df.columns = ['Year', 'Return (%)']

In [ ]:
px.bar(yearly_df, 
      x = 'Year',
      y = 'Return (%)',
      title = 'Bitcoin Yearly Returns-Best vs Worst Years')